### Step 1: Import libraries and setup API

In [47]:
import os
from openai import OpenAI
from IPython.display import display, Markdown
import gradio as gr 
from dotenv import load_dotenv
import json


In [48]:
#loading in environment
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception ( "API Key is missing")
else:
    print(OPENAI_API_KEY[:8])

sk-proj-


### Step 2: Simple RAG w/guardrails and dynamic context injection

In [49]:
system_message = """
You are a digital twin of Simrun Sharma.

You answer questions as Simrun using first-person language.

IMPORTANT RULES:

1. ONLY use information contained between *** markers.
2. DO NOT use outside knowledge about Simrun.
3. DO NOT infer facts that are not explicitly stated.
4. DO NOT invent experiences, opinions, accomplishments, preferences, relationships, skills, or goals.
5. If information is not available between the *** markers, respond:

"I don't know based on the information available to me."

6. Treat all information between *** markers as authoritative.
7. Never contradict information found between *** markers.
8. If multiple *** sections are provided, combine them when answering.

The following information is the ONLY information you know about Simrun:

***
Name: Simrun Sharma

Current Role:
Associate Research Analyst / Data Scientist at CNA.

Education:
Master of Data Science student at Duke University.

Career Goals:
Simrun is actively transitioning from Data Science into AI Engineering.

Roles of interest:
- AI Engineer
- Applied AI Engineer
- AI Deployment Engineer
- Forward Deployed AI Engineer
- AI Solutions Engineer

Industries of interest:
- Artificial Intelligence
- Healthcare Technology
- Defense Technology
- Data Science

Professional Interests:
- Artificial Intelligence
- Machine Learning
- Generative AI
- Agentic AI Systems
- Data Science
- Healthcare Analytics
- Brain Computer Interfaces
- Neurotechnology
- Explainable AI

Healthcare:
Healthcare is one of Simrun's strongest passions. She is interested in applying AI and data science to improve patient outcomes, healthcare operations, accessibility, and clinical decision-making.

Learning Style:
Simrun learns best through:
- Step-by-step explanations
- Visual examples
- Interactive discussions
- Hands-on projects
- Building intuition before technical depth

Communication Style:
- Curious
- Analytical
- Direct
- Practical
- Detail-oriented

Work Preferences:
Simrun enjoys solving real-world problems, working with stakeholders, building practical solutions, and seeing the impact of her work.

Personality:
Simrun is curious, ambitious, persistent, analytical, detail-oriented, and growth-focused.
***
"""

In [50]:
#defining the respond_ai function
def respond_ai (message, history):
    client = OpenAI(api_key=OPENAI_API_KEY)

    response = client.chat.completions.create(
        messages= [{"role":"system", "content" : system_message}] + history + [{"role" : "user", "content" : message}],
        model = "gpt-4.1-mini"
    )

    reply  = response.choices[0].message.content

    return reply

#### Dynamic Context Injection

In [51]:
topic_context = {
    "pineapple": """
***
Food Preference:
Simrun likes pineapple on pizza.
***
""",

    "pickleball": """
***
Hobby:
Simrun has been trying to get into pickleball since moving to Arlington, Virginia. She has taken beginner classes and is working toward becoming an intermediate player.
***
""",

    "dance": """
***
Dance:
Simrun participates on a weekend dance team and enjoys Bachata, Salsa, and Bollywood dance.
***
""",

    "fitness": """
***
Fitness:
Simrun works with a personal trainer and is focused on becoming stronger and more athletic.
***
"""
}

#### System Enhanced prompt

In [52]:
def respond_system_enhanced (message, history):

    system_message_enhanced = system_message
    for keyword, context in topic_context.items():
        if keyword in message.lower():
            system_message_enhanced = system_message_enhanced + "\n\n" + context
    
    client = OpenAI()

    # print(f"system_message_enhanced, {system_message_enhanced}")
    response = client.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = [{"role" : "system", "content": system_message_enhanced}] + history + [{"role" : "user", "content" : message}]

    )

    reply = response.choices[0].message.content

    return reply

### Step 3: Prepare the list of tools for the LLM:

In [53]:
tools = []

### Step 3A: Adding tools : Pushover

In [54]:
#PUSHOVER CREDENTIALS
#Creating the pushover identification information
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = 'https://api.pushover.net/1/messages.json'

# print(pushover_user)

#creating the def send notifications:
import requests

def send_notifications(message:str):
    payload = {'user' : pushover_user,"token" : pushover_token, 'message': message}
    requests.post(url = pushover_url, data=payload)

#Testing send notification
# send_notifications('June 8, 2026 Three Whistles')

# This is not a dictionary. This is a description card written in JSON format.
# We are describing the tool to OpenAI so it knows how to use it.
# Think of it like a form or a menu card. Not real Python objects.
#DESCRIBE PUSHOVER AS AN LLM TOOL
send_notifications_function = {

    # The name of the tool OpenAI will request when it wants to use it
    'name': 'send_notifications',

    # When should OpenAI use this tool? This description tells it.
    'description': 'Sends a push notification to the real-world version of you via Pushover on mobile. Use this if the user needs to alert the real-world version of you',
    'parameters': {

        # Always object. This is a package/container.
        # Even if we only have one input now, 
        # we wrap it in a package in case we add more inputs later.
        'type': 'object',

        # Here are the individual inputs inside the package.
        # Each one described separately so OpenAI knows what to fill in.
        'properties': {

            # First input is called message.
            'message': {

                # It must be text. Not a number. Not a list. Just text.
                'type': 'string',

                # This tells OpenAI what to actually write here.
                'description': 'The notification message the user wants sent to their device'
            }

            # If we needed more inputs later we would add them here.
            # 'username': { 'type': 'string', 'description': '...' },
            # 'priority': { 'type': 'string', 'description': '...' }
        },

        # These inputs cannot be skipped. OpenAI must fill them in.
        # It is a list because there could be multiple required inputs.
        #Could be later ['messsage','username','priority']
        'required': ['message']
    }
}

#adding this tool to the tool list:
# tools = [{'type' : 'function', 'function': send_notifications}] --- I wrote this and it was wrong because the function is the description card 
# not the function itself when you are doing tool calling
tools.append({"type" : "function", "function":send_notifications_function})

### Step 3B: Add Dice rolling

In [55]:
import random
#creating roll dice function
def dice_roll():
    result = random.randint(1,6)
    return result

#creating roll dice description of tool for LLM:
roll_dice_function = {
    'name' : "dice_roll",
    'description': 'This is a function where you can roll a dice and can see the number you get from the roll',
    "parameters": {

    },
    "required" : []
}

#appending this to our available tools
tools.append({"type" : "function", "function" : roll_dice_function})

### Step 4: Function to handle LLM Tool Calls

In [56]:
def handle_tool_call(tool_calls:list):
        

        tool_call_results = []

        for tool_call in tool_calls:

            function_name = tool_call.function.name

            #  --- for debugging
            # print(f"the function name is {function_name}")
                
            # tool_call = tools_calls[0] #this is because we only have one tool thus far
            
            args = json.loads(tool_call.function.arguments)

            if function_name == 'send_notifications':
                send_notifications(args['message']) #sent to pushover

                content = f"Sent Notification: {args['message']}"

            elif function_name == "dice_roll":

                content =  f' Rolled: {dice_roll()}'

            # elif function_name == "insert_function3_name":

            #     content =insert_function3_name (args['message']})

            else:
                 content = f"Unknown function: {function_name}"

            #package into a dictionary for the llm to see what we got out from the tool call
            tool_call_result = {
            'role' : 'tool',
            'content' : content ,
                'tool_call_id' : tool_call.id   
            }

            print(f"this is what the tool_call_result from handle tool_call looks like: {tool_call_result}")

            #appending each tool_call_result to the tool_call_results list of dictionaries
            tool_call_results.append(tool_call_result)

        return tool_call_results

### Step 5: Function to Process the Conversation Turn 

In [57]:
def respond_system_enhanced (message, history):

    system_message_enhanced = system_message
    for keyword, context in topic_context.items():
        if keyword in message.lower():
            system_message_enhanced = system_message_enhanced + "\n\n" + context
    
    client = OpenAI()

    messages = [{"role" : "system", "content": system_message_enhanced}] + history + [{"role" : "user", "content" : message}]

    # print(f"system_message_enhanced, {system_message_enhanced}")
    #give the tool to the model
    response = client.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages,
        tools=tools,
        tool_choice= 'auto'
    )

    
    message = response.choices[0].message

    print("message before while loop", message)


    
    while message.tool_calls:
        from pprint import pprint
        pprint(message.tool_calls)
        #we want to pass the list of tool calls to out=r handle tool call function
        tool_call = message.tool_calls
        tool_call_result = handle_tool_call(message.tool_calls)

        #we want to append the message with the user request and the first two tool call requests
        messages.append(message)
        #We also want append to append the toolcall results with the dictionaries of the content return from each tool call
        messages.extend(tool_call_result)

        response = client.chat.completions.create(
            model = "gpt-4.1-mini",
            messages = messages,
            tools=tools
        )

        message = response.choices[0].message

    return(message.content)


    

### Step 6: Launch Gradio

In [58]:
#launching the browser

gr.ChatInterface(fn = respond_system_enhanced).launch(inbrowser=False)

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


message before while loop ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_lAsZCbbliO3jMIS3WraimGFQ', function=Function(arguments='{}', name='dice_roll'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_W3526OpJoyG4KrAkvVqezWGe', function=Function(arguments='{"message": "The highest role is AI Engineer"}', name='send_notifications'), type='function')])
[ChatCompletionMessageFunctionToolCall(id='call_lAsZCbbliO3jMIS3WraimGFQ', function=Function(arguments='{}', name='dice_roll'), type='function'),
 ChatCompletionMessageFunctionToolCall(id='call_W3526OpJoyG4KrAkvVqezWGe', function=Function(arguments='{"message": "The highest role is AI Engineer"}', name='send_notifications'), type='function')]
this is what the tool_call_result from handle tool_call looks like: {'role': 'tool', 'content': ' Rolled: 6', 'tool_call_id': 'call_lAsZCbbliO3jMIS3WraimGFQ'